In [1]:
!pip install -q llama-index

In [3]:
import sys
import time
import pickle
import logging
from pathlib import Path
from typing import List, Optional, Tuple, Union
import numpy as np
from scipy.sparse import csr_matrix, save_npz, load_npz
from llama_index.core import Document
# Add parent directory to Python path
# Assuming your project root is `/work`, add it to the system path
project_root = Path('/work')
sys.path.append(str(project_root))
from src.utils.data_utils import load_documents, select_random_documents
from src.utils.evaluation import (
    perform_analysis,
    perform_single_match_analysis,
    print_single_match_analysis,
    print_sample_results,
    calculate_prefix_success_at_k,
    perform_prefix_match_analysis,
    print_prefix_match_analysis
)
from contextlib import contextmanager

# ---------------------------
# Timing context manager
# ---------------------------
@contextmanager
def timer(name):
    omitted_names = {
        "Selecting a random document for sanity check",
        "Encoding query",
        "Searching in Milvus",
        "Processing search results"
    }
    start = time.time()
    yield
    end = time.time()
    if name not in omitted_names and not name.startswith("Retrieval for document"):
        print(f"[Timing] {name}: {end - start:.4f} seconds")

logging.basicConfig(level=logging.INFO)

def main(index_path_texto, index_path_resumen, file_identifier, batch_size=500, top_k=10):
    start_total = time.time()

    # Load persisted documents
    with open(index_path_texto / "documents.pkl", "rb") as f:
        documents_texto = pickle.load(f)
    with open(index_path_resumen / "documents.pkl", "rb") as f:
        documents_resumen = pickle.load(f)

    # Load persisted vectorizers and embeddings
    with open(index_path_texto / "vectorizer.pkl", "rb") as f:
        vectorizer_texto = pickle.load(f)
    embeddings_texto = load_npz(index_path_texto / "embeddings.npz")

    with open(index_path_resumen / "vectorizer.pkl", "rb") as f:
        vectorizer_resumen = pickle.load(f)
    # (We won’t use vectorizer_resumen here since we’re reusing vectorizer_texto for retrieval.)

    # Build a mapping from texto item keys to their indices for fast lookup
    texto_mapping = {doc.metadata.get("item_key"): idx for idx, doc in enumerate(documents_texto)}

    # Select a random sample from the resumen documents
    sample_size = min(len(documents_resumen), 16590)
    random_resumen = select_random_documents(documents_resumen, sample_size)
    resumen_keys, resumen_texts = zip(*random_resumen)  # unzip into separate lists

    all_results = []
    total_batches = (sample_size + batch_size - 1) // batch_size
    start_batches = time.time()

    # Batch processing: transform and compute scores in batches
    for batch_num, start_idx in enumerate(range(0, sample_size, batch_size), 1):
        end_idx = min(start_idx + batch_size, sample_size)
        batch_keys = resumen_keys[start_idx:end_idx]
        batch_texts = resumen_texts[start_idx:end_idx]

        # Batch-transform resumen texts using vectorizer_texto
        batch_query_embeddings = vectorizer_texto.transform(batch_texts)
        # Compute similarity scores in one sparse matrix multiplication
        # scores_batch: shape (batch_size, number of texto documents)
        scores_batch = (batch_query_embeddings @ embeddings_texto.T).toarray()

        # Process each query in the batch
        for i, score_row in enumerate(scores_batch):
            overall_idx = start_idx + i
            item_key = batch_keys[i]
            resumen_text = batch_texts[i]

            # Extract top_k indices and sort them by score descending
            topk_idx_unsorted = np.argpartition(score_row, -top_k)[-top_k:]
            topk_idx = topk_idx_unsorted[np.argsort(score_row[topk_idx_unsorted])[::-1]]

            # Build the list of top_k results
            texto_results = [{
                "item_key": documents_texto[idx].metadata.get("item_key", f"Doc-{idx}"),
                "text": documents_texto[idx].text,
                "score": score_row[idx]
            } for idx in topk_idx]

            # Get target score from matching texto document (if exists)
            texto_index = texto_mapping.get(item_key, None)
            if texto_index is not None:
                target_score = score_row[texto_index]
                target_text = documents_texto[texto_index].text
            else:
                target_score = 0.0
                target_text = "No matching texto found"

            # Append results for this resumen document
            all_results.append({
                'resumen_item_key': item_key,
                'resumen_text': resumen_text,
                'target_text': target_text,
                'target_score': target_score,
                'texto_results': texto_results
            })

        # Logging progress for each batch
        elapsed_batches = time.time() - start_batches
        batches_remaining = total_batches - batch_num
        avg_batch_time = elapsed_batches / batch_num
        estimated_remaining_sec = batches_remaining * avg_batch_time
        est_min, est_sec = divmod(estimated_remaining_sec, 60)
        logging.info(
            f"Batch {batch_num}/{total_batches} processed. "
            f"Estimated time to completion: {int(est_min)}m {int(est_sec)}s"
        )

    # Perform overall analysis
    analysis_results = perform_analysis(all_results)
    print("\nAnalysis Results:")
    for metric, value in analysis_results.items():
        print(f"{metric}: {value:.4f}")

    # Perform single-match analysis and print results
    metrics = perform_single_match_analysis(all_results, top_k=top_k)
    print_single_match_analysis(metrics, top_k=top_k)

    # Perform prefix-matching analysis
    with timer("Performing prefix-matching analysis"):
        prefix_length = 6  # Match first 6 characters of item_key
        prefix_match_metrics = perform_prefix_match_analysis(
            all_results, 
            top_k=10, 
            prefix_length=prefix_length
        )
        print_prefix_match_analysis(
            prefix_match_metrics, 
            top_k=10, 
            prefix_length=prefix_length
        )

    # Print some sample retrieval results
    print_sample_results(all_results)

    total_elapsed = time.time() - start_total
    tot_min, tot_sec = divmod(total_elapsed, 60)
    print(f"\nTotal execution time: {int(tot_min)}m {int(tot_sec)}s")


In [4]:
index_path_texto = Path(f'/work/sandbox/bow/indexes/texto')
index_path_resumen = Path(f'/work/sandbox/bow/indexes/resumen')
main(index_path_texto, index_path_resumen, 'OEB')

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 28s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 25s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 24s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 23s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 22s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 21s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 20s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 19s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 18s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 17s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 16s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 15s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 15s
INFO:root:Batch 15/34 process


Analysis Results:
Recall: 0.0004
Precision: 0.0000
F1 Score: 0.0001
Mean Reciprocal Rank (MRR): 0.0002
Mean Average Precision (MAP): 0.0002

Single-Match Analysis Results:
Success@10             : 0.04%
MRR: 0.0002
Mean Position: 4.33
Median Position: 5.5
Perfect Matches: 0.01%

Position Distribution:
------------------------------
Position_1: 0.01%
Position_2: 0.00%
Position_3: 0.00%
Position_4: 0.00%
Position_5: 0.01%
Position_6: 0.01%
Position_7: 0.01%
Position_8: 0.00%
Position_9: 0.00%
Position_10: 0.00%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 0.04%
Prefix_MRR: 0.0004
Prefix_Mean_Position: 1.00
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 0.04%

Prefix Position Distribution:
----------------------------------------
Position 1    : 0.04%
Position 2    : 0.00%
Position 3    : 0.00%
Position 4    : 0.00%
Position 5    : 0.00%
Position 6    : 0.00%
Position 7    : 0.00%
Position 8    : 0.00%
Position 9    : 0.00%
Position 10   : 

## Inspection of the BoW index

In [5]:
import pickle
import numpy as np
from scipy.sparse import load_npz
from pathlib import Path
import pandas as pd

def inspect_bow_index(index_path, num_documents=5, num_features=20):
    """
    Inspect a Bag-of-Words index to understand its structure and content
    """
    print(f"Inspecting BoW index at: {index_path}")
    print("=" * 60)
    
    # Load the components
    with open(index_path / "vectorizer.pkl", "rb") as f:
        vectorizer = pickle.load(f)
    
    embeddings = load_npz(index_path / "embeddings.npz")
    
    with open(index_path / "documents.pkl", "rb") as f:
        documents = pickle.load(f)
    
    # Basic statistics
    print(f"📊 BASIC STATISTICS")
    print(f"Number of documents: {len(documents)}")
    print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
    print(f"Embeddings shape: {embeddings.shape}")
    print(f"Matrix density: {embeddings.nnz / (embeddings.shape[0] * embeddings.shape[1]):.6f}")
    print(f"Total non-zero entries: {embeddings.nnz:,}")
    print()
    
    # Vocabulary inspection
    print(f"🔤 VOCABULARY SAMPLE")
    vocab_items = list(vectorizer.vocabulary_.items())
    vocab_sample = sorted(vocab_items)[:num_features]  # First 20 alphabetically
    for word, idx in vocab_sample:
        print(f"  '{word}' -> index {idx}")
    print(f"  ... and {len(vocab_items) - num_features} more terms")
    print()
    
    # Most frequent terms across all documents
    print(f"📈 MOST FREQUENT TERMS (across all documents)")
    term_frequencies = np.array(embeddings.sum(axis=0)).flatten()
    feature_names = vectorizer.get_feature_names_out()
    
    # Get top terms
    top_indices = term_frequencies.argsort()[-num_features:][::-1]
    for i, idx in enumerate(top_indices, 1):
        term = feature_names[idx]
        freq = term_frequencies[idx]
        print(f"  {i:2d}. '{term}': {freq:,} occurrences")
    print()
    
    # Document-level inspection
    print(f"📄 DOCUMENT SAMPLES")
    for i in range(min(num_documents, len(documents))):
        doc = documents[i]
        doc_text = doc.text if hasattr(doc, 'text') else str(doc)
        
        print(f"\nDocument {i+1}:")
        print(f"  Preview: {doc_text[:100]}...")
        
        # Get BoW representation for this document
        doc_vector = embeddings[i].toarray().flatten()
        non_zero_indices = np.nonzero(doc_vector)[0]
        
        print(f"  Non-zero terms: {len(non_zero_indices)}")
        print(f"  Total word count: {doc_vector.sum():.0f}")
        
        # Show top terms in this document
        if len(non_zero_indices) > 0:
            doc_term_counts = [(feature_names[idx], doc_vector[idx]) for idx in non_zero_indices]
            doc_term_counts.sort(key=lambda x: x[1], reverse=True)
            
            print(f"  Top terms in this document:")
            for term, count in doc_term_counts[:10]:  # Top 10 terms
                print(f"    '{term}': {count:.0f}")
    print()

def compare_document_vectors(index_path, doc_indices=[0, 1]):
    """
    Compare BoW vectors between specific documents
    """
    print(f"🔍 COMPARING DOCUMENTS")
    print("=" * 40)
    
    # Load components
    with open(index_path / "vectorizer.pkl", "rb") as f:
        vectorizer = pickle.load(f)
    
    embeddings = load_npz(index_path / "embeddings.npz")
    feature_names = vectorizer.get_feature_names_out()
    
    with open(index_path / "documents.pkl", "rb") as f:
        documents = pickle.load(f)
    
    for doc_idx in doc_indices:
        if doc_idx >= len(documents):
            print(f"Document {doc_idx} doesn't exist (only {len(documents)} documents)")
            continue
            
        doc = documents[doc_idx]
        doc_text = doc.text if hasattr(doc, 'text') else str(doc)
        doc_vector = embeddings[doc_idx].toarray().flatten()
        
        print(f"\nDocument {doc_idx}:")
        print(f"  Text preview: {doc_text[:150]}...")
        print(f"  Vector summary: {len(np.nonzero(doc_vector)[0])} unique terms, {doc_vector.sum():.0f} total words")
        
        # Show non-zero entries
        non_zero_indices = np.nonzero(doc_vector)[0]
        if len(non_zero_indices) > 0:
            terms_and_counts = [(feature_names[idx], doc_vector[idx]) for idx in non_zero_indices]
            terms_and_counts.sort(key=lambda x: x[1], reverse=True)
            
            print(f"  BoW representation (top 15 terms):")
            for term, count in terms_and_counts[:15]:
                print(f"    {term}: {count:.0f}")

def show_bow_as_dataframe(index_path, max_docs=5, max_features=20):
    """
    Show BoW as a DataFrame for easy viewing
    """
    print(f"📋 BAG-OF-WORDS AS TABLE")
    print("=" * 40)
    
    # Load components
    with open(index_path / "vectorizer.pkl", "rb") as f:
        vectorizer = pickle.load(f)
    
    embeddings = load_npz(index_path / "embeddings.npz")
    feature_names = vectorizer.get_feature_names_out()
    
    # Get most frequent terms for display
    term_frequencies = np.array(embeddings.sum(axis=0)).flatten()
    top_indices = term_frequencies.argsort()[-max_features:][::-1]
    top_features = [feature_names[idx] for idx in top_indices]
    
    # Create DataFrame
    dense_matrix = embeddings[:max_docs, top_indices].toarray()
    df = pd.DataFrame(dense_matrix, 
                     columns=top_features,
                     index=[f"Doc_{i}" for i in range(max_docs)])
    
    print(f"BoW matrix (first {max_docs} docs, top {max_features} terms):")
    print(df)
    return df


INFO:numexpr.utils:Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
INFO:numexpr.utils:NumExpr defaulting to 8 threads.


In [8]:
index_path = Path('/work/sandbox/BoW/indexes/texto')
inspect_bow_index(index_path, num_documents=5, num_features=20)

Inspecting BoW index at: /work/sandbox/BoW/indexes/texto
📊 BASIC STATISTICS
Number of documents: 47514
Vocabulary size: 326
Embeddings shape: (47514, 326)
Matrix density: 0.173278
Total non-zero entries: 2,684,003

🔤 VOCABULARY SAMPLE
  '0' -> index 0
  '00' -> index 1
  '1' -> index 2
  '10' -> index 3
  '100mm' -> index 4
  '110' -> index 5
  '12' -> index 6
  '150' -> index 7
  '16' -> index 8
  '160' -> index 9
  '18' -> index 10
  '2' -> index 11
  '20' -> index 12
  '200' -> index 13
  '21mm' -> index 14
  '220' -> index 15
  '24' -> index 16
  '250' -> index 17
  '29mm' -> index 18
  '3' -> index 19
  ... and 306 more terms

📈 MOST FREQUENT TERMS (across all documents)
   1. 'de': 656,872 occurrences
   2. 'y': 242,624 occurrences
   3. 'la': 200,548 occurrences
   4. 'el': 164,850 occurrences
   5. 'los': 149,805 occurrences
   6. 'tubos': 88,606 occurrences
   7. 'ejecución': 66,408 occurrences
   8. 'suministro': 54,234 occurrences
   9. 'montaje': 53,694 occurrences
  10. 'd

In [9]:
compare_document_vectors(index_path, doc_indices=[0, 1, 2])

🔍 COMPARING DOCUMENTS

Document 0:
  Text preview: ZANJAS, CANALIZACIONES Y TUBOS...
  Vector summary: 4 unique terms, 4 total words
  BoW representation (top 15 terms):
    canalizaciones: 1
    tubos: 1
    y: 1
    zanjas: 1

Document 1:
  Text preview: Ejecución de canalización para línea subterránea de doble circuito de 220 o 400 kV de cables entubados y tubos embebidos en hormigón HM-20, de dimensi...
  Vector summary: 110 unique terms, 207 total words
  BoW representation (top 15 terms):
    de: 35
    la: 9
    mm: 9
    diámetro: 6
    para: 6
    tubos: 6
    y: 6
    2: 4
    a: 3
    ejecución: 3
    el: 3
    exterior: 3
    m: 3
    polietileno: 3
    110: 2

Document 2:
  Text preview: Ejecución de canalización para línea subterránea de doble circuito de 220 o 400 kV de cables entubados y tubos embebidos en hormigón HM-20, de dimensi...
  Vector summary: 110 unique terms, 207 total words
  BoW representation (top 15 terms):
    de: 35
    la: 9
    mm: 9
    diámetro: 6


In [11]:
df = show_bow_as_dataframe(index_path, max_docs=5, max_features=25)
print(df)

📋 BAG-OF-WORDS AS TABLE
BoW matrix (first 5 docs, top 25 terms):
       de  y  la  el  los  tubos  ejecución  suministro  montaje  diámetro  \
Doc_0   0  1   0   0    0      1          0           0        0         0   
Doc_1  35  6   9   3    2      6          3           1        0         6   
Doc_2  35  6   9   3    2      6          3           1        0         6   
Doc_3  36  6   9   3    2      6          4           1        0         6   
Doc_4  35  6   9   3    2      6          3           1        0         6   

       ...  banda  canalización  al  transporte  lugar  relleno  compactado  \
Doc_0  ...      0             0   0           0      0        0           0   
Doc_1  ...      0             1   1           1      0        1           1   
Doc_2  ...      0             1   1           1      0        1           1   
Doc_3  ...      0             1   1           1      0        1           1   
Doc_4  ...      0             1   1           1      0        1        

In [12]:
index_path_texto = Path(f'/work/sandbox/bow/indexes/texto_Restricted')
index_path_resumen = Path(f'/work/sandbox/bow/indexes/resumen_Restricted')
main(index_path_texto, index_path_resumen, 'OEB')

INFO:root:Batch 1/34 processed. Estimated time to completion: 0m 17s
INFO:root:Batch 2/34 processed. Estimated time to completion: 0m 16s
INFO:root:Batch 3/34 processed. Estimated time to completion: 0m 15s
INFO:root:Batch 4/34 processed. Estimated time to completion: 0m 14s
INFO:root:Batch 5/34 processed. Estimated time to completion: 0m 14s
INFO:root:Batch 6/34 processed. Estimated time to completion: 0m 14s
INFO:root:Batch 7/34 processed. Estimated time to completion: 0m 13s
INFO:root:Batch 8/34 processed. Estimated time to completion: 0m 12s
INFO:root:Batch 9/34 processed. Estimated time to completion: 0m 12s
INFO:root:Batch 10/34 processed. Estimated time to completion: 0m 12s
INFO:root:Batch 11/34 processed. Estimated time to completion: 0m 11s
INFO:root:Batch 12/34 processed. Estimated time to completion: 0m 10s
INFO:root:Batch 13/34 processed. Estimated time to completion: 0m 10s
INFO:root:Batch 14/34 processed. Estimated time to completion: 0m 9s
INFO:root:Batch 15/34 processe


Analysis Results:
Recall: 0.0108
Precision: 0.0011
F1 Score: 0.0020
Mean Reciprocal Rank (MRR): 0.0031
Mean Average Precision (MAP): 0.0031

Single-Match Analysis Results:
Success@10             : 1.08%
MRR: 0.0031
Mean Position: 5.48
Median Position: 5.0
Perfect Matches: 0.09%

Position Distribution:
------------------------------
Position_1: 0.09%
Position_2: 0.13%
Position_3: 0.10%
Position_4: 0.16%
Position_5: 0.08%
Position_6: 0.08%
Position_7: 0.10%
Position_8: 0.15%
Position_9: 0.08%
Position_10: 0.11%

Prefix Matching Analysis Results (first 6 characters):
Prefix_Success@10             : 2.89%
Prefix_MRR: 0.0218
Prefix_Mean_Position: 2.30
Prefix_Median_Position: 1.0
Prefix_Perfect_Matches: 1.89%

Prefix Position Distribution:
----------------------------------------
Position 1    : 1.89%
Position 2    : 0.33%
Position 3    : 0.13%
Position 4    : 0.03%
Position 5    : 0.06%
Position 6    : 0.08%
Position 7    : 0.19%
Position 8    : 0.08%
Position 9    : 0.08%
Position 10   : 

In [13]:
index_path = Path('/work/sandbox/BoW/indexes/texto_Restricted')
inspect_bow_index(index_path, num_documents=5, num_features=20)

Inspecting BoW index at: /work/sandbox/BoW/indexes/texto_Restricted
📊 BASIC STATISTICS
Number of documents: 47514
Vocabulary size: 153
Embeddings shape: (47514, 153)
Matrix density: 0.141064
Total non-zero entries: 1,025,486

🔤 VOCABULARY SAMPLE
  '0' -> index 0
  '0,80 a 1,00 m' -> index 1
  '1' -> index 2
  '1 o 2' -> index 3
  '1''' -> index 4
  '1,10 m' -> index 5
  '110' -> index 6
  '110mm' -> index 7
  '12' -> index 8
  '16' -> index 9
  '160mm' -> index 10
  '18' -> index 11
  '2' -> index 12
  '2"' -> index 13
  '2''' -> index 14
  '20' -> index 15
  '200mm' -> index 16
  '21mm' -> index 17
  '220' -> index 18
  '24' -> index 19
  ... and 133 more terms

📈 MOST FREQUENT TERMS (across all documents)
   1. 'tubos': 88,606 occurrences
   2. 'ejecución': 66,408 occurrences
   3. 'suministro': 54,234 occurrences
   4. 'diámetro': 50,940 occurrences
   5. 'mm': 48,494 occurrences
   6. 'condiciones': 47,484 occurrences
   7. 'banda': 47,454 occurrences
   8. 'mantenimiento': 47,454 

In [14]:
df = show_bow_as_dataframe(index_path, max_docs=5, max_features=25)
print(df)

📋 BAG-OF-WORDS AS TABLE
BoW matrix (first 5 docs, top 25 terms):
       tubos  ejecución  suministro  diámetro  mm  condiciones  banda  \
Doc_0      1          0           0         0   0            0      0   
Doc_1      6          3           1         6   9            1      0   
Doc_2      6          3           1         6   9            1      0   
Doc_3      6          4           1         6   9            1      0   
Doc_4      6          3           1         6   9            1      0   

       mantenimiento  trabajo  canalización  ...  hormigonada  polietileno  3  \
Doc_0              0        0             0  ...            0            0  0   
Doc_1              0        0             1  ...            0            3  1   
Doc_2              0        0             1  ...            0            3  1   
Doc_3              0        0             1  ...            0            3  1   
Doc_4              0        0             1  ...            0            3  1   

       5 